# 20a. Phân tích các đợt tấn công không phát hiện (GĐ J, bổ sung task 8.4), CPU
Đọc `windows.csv` và `detector_summary.json` của notebook 20 (hybrid chính, seed 42); tính lại điểm tấn công từng node bằng mô hình đã đóng băng
(đường offline, khớp 100% với online); với từng đợt không có cảnh báo ghi loại tấn công, số flow, số node, độ dài, tỉ lệ node chỉ nhận flow
và lý do (điểm dưới ngưỡng, bị gán Normal, đợt quá ngắn). Không đổi mô hình, ngưỡng, dự đoán.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "không rõ"


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
INFO["cpu"] = cpu_model()
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| CPU:", INFO["cpu"])

## Đường dẫn đầu vào (như notebook 20, thêm output của notebook 20)

In [ ]:
DS = "nf_ton_iot_v3"
OLD = f"{REPO}/reports/kaggle"
SPEC = {DS: dict(data=os.path.dirname(find(f"{DS}/split.json")),
                 ckpt=os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt")),
                 folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS}_s42/lgbm_oofavg_il.pkl"))),
                 light=os.path.dirname(os.path.dirname(find(f"models_light/{DS}_s42/lgbm_n100_l15.pkl"))),
                 baseline_search=f"{OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS}.json",
                 ablation_search=f"{OLD}/gnn-iot-ids-16d-ablation-ton/ablation_search_{DS}.json",
                 gnn_csv=f"{OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS}.csv",
                 hybrid_csv=f"{OLD}/gnn-iot-ids-15f-hybrid-il-ton/results_hybrid_oofavg_il_{DS}.csv")}
CFG = dict(dataset=DS, split="test", flows_csv=find("NF-ToN-IoT-v3.csv"), expect_commit=INFO["commit"][:7],
           thresholds=f"{OLD}/gnn-iot-ids-16c-aggregate/thresholds.json",
           thresholds_c={DS: f"{OLD}/gnn-iot-ids-16h-c-threshold/thresholds_c_{DS}.json"}, datasets=SPEC)
for k, v in SPEC[DS].items():
    assert os.path.exists(v), f"{k}: không thấy {v}"
for p in (CFG["thresholds"], *CFG["thresholds_c"].values(), CFG["flows_csv"]):
    assert os.path.exists(p), p
json.dump(CFG, open(f"{WORK}/detector_config.json", "w"), ensure_ascii=False, indent=1)
INFO["config"] = CFG
save_info()

RT = os.path.dirname(find("gnn-iot-ids-20-realtime/windows.csv"))
print("Output notebook 20:", RT, os.listdir(RT))

## Phân tích

In [ ]:
with step("undetected_episodes"):
    run([sys.executable, "-I", "scripts/undetected_episodes.py", "--config", f"{WORK}/detector_config.json", "--model", "hybrid_gat_lgbm_oofavg_il",
         "--windows-csv", f"{RT}/windows.csv", "--summary-json", f"{RT}/detector_summary.json", "--out", f"{WORK}/undetected_episodes.json",
         "--flows-cache", "/tmp/flows_test.parquet"], log=f"{WORK}/undetected_episodes_log.txt", cwd=REPO)
res = json.load(open(f"{WORK}/undetected_episodes.json", encoding="utf-8"))
INFO["result"] = {k: res[k] for k in ("n_episodes", "n_detected", "n_undetected", "consistent_with_realtime_eval", "summary")}
save_info()
print(json.dumps(INFO["result"], indent=1, ensure_ascii=False))